# 01 — Ingest & Validate

Verify all parquets are built and match expected shape/date range.
Run the build scripts first if parquets are missing:
```
python src/processing/build_ontime_dmv.py
python src/processing/build_t100_dmv.py
python src/ingestion/db1b_download.py   # downloads + filters quarterly files
python src/processing/build_db1b_dmv.py
```

In [1]:
import pandas as pd
from pathlib import Path

import sys
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from config import PROCESSED_DATA_PATH

PROC = PROJECT_ROOT / PROCESSED_DATA_PATH

In [2]:
ontime = pd.read_parquet(PROC / 'ontime_dmv.parquet')
ontime['FlightDate'] = pd.to_datetime(ontime['FlightDate'])
print('on-time:', ontime.shape)
print('  date range:', ontime['FlightDate'].min().date(), '->', ontime['FlightDate'].max().date())
print('  airports:', sorted(ontime['Origin'].unique()))
print()

on-time: (2823044, 33)
  date range: 2015-01-01 -> 2026-01-31
  airports: ['BWI', 'DCA', 'IAD']



In [3]:
t100 = pd.read_parquet(PROC / 't100_intl_dmv.parquet')
print('T-100 intl:', t100.shape)
print('  year range:', t100['YEAR'].min(), '->', t100['YEAR'].max())
print('  airports (ORIGIN):', sorted(t100[t100['ORIGIN'].isin(['IAD','DCA','BWI'])]['ORIGIN'].unique()))
print('  top dest countries:', t100['DEST_COUNTRY'].value_counts().head(8).to_dict())

T-100 intl: (35846, 23)
  year range: 2015 -> 2026
  airports (ORIGIN): ['BWI', 'DCA', 'IAD']
  top dest countries: {'US': 18268, 'CA': 2705, 'MX': 1376, 'GB': 1376, 'DE': 1053, 'FR': 600, 'DO': 579, 'SV': 543}


In [4]:
# Null audit — ontime delay cause columns
delay_cols = ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
print('on-time null rates:')
print(ontime[delay_cols].isnull().mean().round(3))

on-time null rates:
CarrierDelay         0.808
WeatherDelay         0.808
NASDelay             0.808
SecurityDelay        0.808
LateAircraftDelay    0.808
dtype: float64


## DB1B fare data

In [5]:
db1b = pd.read_parquet(PROC / 'db1b_dmv.parquet')
print('DB1B:', db1b.shape)
print('  quarter range:', f"{db1b['Year'].min()} Q{db1b[db1b['Year']==db1b['Year'].min()]['Quarter'].min()}",
      '->', f"{db1b['Year'].max()} Q{db1b[db1b['Year']==db1b['Year'].max()]['Quarter'].max()}")
print('  airports (Origin):', sorted(db1b['Origin'].unique()))
print()
print('MktFare stats (non-bulk, non-zero):')
fare = db1b[(db1b['BulkFare'] == 0) & (db1b['MktFare'] > 0)]['MktFare']
print(fare.describe().round(2))
print()
print('null rates:')
print(db1b.isnull().mean().round(3).to_string())

DB1B: (19936926, 23)


  quarter range: 2015 Q1 -> 2025 Q2


  airports (Origin): ['ABE', 'ABI', 'ABQ', 'ABR', 'ABY', 'ACK', 'ACT', 'ACV', 'ACY', 'ADK', 'ADQ', 'AEX', 'AGS', 'AIA', 'AIN', 'AKN', 'ALB', 'ALO', 'ALS', 'ALW', 'AMA', 'ANC', 'APN', 'ART', 'ASE', 'ATL', 'ATW', 'ATY', 'AUG', 'AUS', 'AVL', 'AVP', 'AZO', 'BDL', 'BET', 'BFD', 'BFF', 'BFL', 'BGM', 'BGR', 'BHB', 'BHM', 'BIH', 'BIL', 'BIS', 'BJI', 'BKG', 'BKW', 'BLI', 'BMI', 'BNA', 'BOI', 'BOS', 'BPT', 'BQK', 'BQN', 'BRD', 'BRL', 'BRO', 'BRW', 'BTM', 'BTR', 'BTV', 'BUF', 'BUR', 'BWI', 'BZN', 'CAE', 'CAK', 'CDB', 'CDC', 'CDR', 'CDV', 'CEC', 'CEZ', 'CGI', 'CHA', 'CHO', 'CHS', 'CID', 'CIU', 'CKB', 'CLD', 'CLE', 'CLL', 'CLT', 'CMH', 'CMI', 'CMX', 'CNM', 'CNY', 'COD', 'COS', 'COU', 'CPR', 'CPX', 'CRP', 'CRW', 'CSG', 'CVG', 'CVN', 'CWA', 'CYS', 'DAB', 'DAL', 'DAY', 'DBQ', 'DCA', 'DDC', 'DEC', 'DEN', 'DFW', 'DHN', 'DIK', 'DLG', 'DLH', 'DRO', 'DRT', 'DSM', 'DTW', 'DUJ', 'DUT', 'DVL', 'EAR', 'EAT', 'EAU', 'ECP', 'EGE', 'EKO', 'ELD', 'ELM', 'ELP', 'ENA', 'ERI', 'ESC', 'EUG', 'EVV', 'EWN', 'EWR', 'EYW'

count    19811058.00
mean          255.44
std           219.64
min             0.09
25%           147.00
50%           216.20
75%           314.50
max        371800.00
Name: MktFare, dtype: float64

null rates:


Year               0.0
Quarter            0.0
Origin             0.0
OriginCountry      0.0
OriginState        0.0
OriginStateName    0.0
OriginWac          0.0
Dest               0.0
DestCountry        0.0
DestState          0.0
DestStateName      0.0
DestWac            0.0
RPCarrier          0.0
TkCarrier          0.0
OpCarrier          0.0
BulkFare           0.0
Passengers         0.0
MktFare            0.0
MktDistance        0.0
MktCoupons         0.0
NonStopMiles       0.0
ItinGeoType        0.0
MktGeoType         0.0
